# MoodIn - ST-GCN (VA-bridge) | Pipeline uji di Kaggle

Memvalidasi pipeline **cabang ekspresi ST-GCN** end-to-end di GPU gratis Kaggle:
ekstraksi landmark -> tensor `(C, T, V)` -> latih (regresi Valence-Arousal, loss CCC) -> evaluasi (CCC, akurasi kuadran mood, macro-F1).

**Baseline = CK+**. Untuk hasil serius, ganti ke **Aff-Wild2** setelah akses EULA disetujui (`--layout affwild2`).

> Mood diklasifikasi SEKALI di ruang V-A (`labels.va_to_mood`) agar tidak konflik dengan cabang STC-rPPG saat fusion.


## 1. Cek lingkungan & GPU

In [ ]:
import torch, sys
print('Python', sys.version.split()[0])
print('Torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Dependensi
Kaggle sudah menyediakan torch. Tambahkan MediaPipe untuk ekstraksi landmark.

In [ ]:
# Aktifkan Internet di panel Settings Kaggle untuk sel ini.
!pip -q install mediapipe==0.10.14 opencv-python==4.10.0.84 pyyaml


## 3. Ambil kode repo
- **A. Clone** dari GitHub (butuh Internet aktif), atau
- **B. Add Data**: unggah repo sebagai Kaggle Dataset lalu salin ke `/kaggle/working`.


In [ ]:
import os, sys
REPO_DIR = '/kaggle/working/MoodIn_Software'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MoodInBME/Software-Architecture.git {REPO_DIR}
    # Opsi B: !cp -r /kaggle/input/moodin-software-architecture {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('Repo di:', REPO_DIR, '| ada model.py:',
      os.path.exists(os.path.join(REPO_DIR, 'models/st_gcn_graph_tran/model.py')))


## 4. Sanity check model (data acak) - pastikan arsitektur jalan dulu

In [ ]:
os.chdir(REPO_DIR)
!python -m models.st_gcn_graph_tran.train --sanity


## 5. Dataset CK+
Tambahkan dataset CK+ lewat **Add Data**. Sesuaikan `CKPLUS_ROOT`.

> Peringatan: banyak mirror CK+ hanya berisi frame puncak (gambar tunggal), bukan urutan.
> Untuk model temporal, pilih mirror ber-urutan frame. Struktur: `root/<emosi>/<klip>/...`


In [ ]:
CKPLUS_ROOT = '/kaggle/input/ckplus'   # <-- sesuaikan
PREP_DIR = '/kaggle/working/prepared/ckplus'
!ls {CKPLUS_ROOT} | head


## 6. Ekstraksi & cache landmark (SEKALI)

In [ ]:
!python data/preprocessing/extract_landmarks_video.py \
    --input_root {CKPLUS_ROOT} --layout ckplus \
    --output_dir {PREP_DIR} --clip_len 150


## 7. Latih (regresi VA, loss CCC)

In [ ]:
!python -m models.st_gcn_graph_tran.train \
    --config models/st_gcn_graph_tran/config.yaml \
    --manifest {PREP_DIR}/manifest.json --val_split 0.2


## 8. Evaluasi dari checkpoint terbaik

In [ ]:
import numpy as np, torch, json
from torch.utils.data import DataLoader
from models.st_gcn_graph_tran.train import build_model
from data.loaders.expression_video_loader import ExpressionLandmarkDataset
from evaluation.va_metrics import evaluate_va

ckpt = torch.load('checkpoints/stgcn_va_best.pth', map_location='cpu')
model = build_model(ckpt['config']); model.load_state_dict(ckpt['model']); model.eval()
ds = ExpressionLandmarkDataset(f'{PREP_DIR}/manifest.json', clip_len=150)
dl = DataLoader(ds, batch_size=16)
P, TVA, TM = [], [], []
with torch.no_grad():
    for x, va, mood in dl:
        P.append(model(x)['va'].numpy()); TVA.append(va.numpy()); TM.append(mood.numpy())
print(json.dumps(evaluate_va(np.concatenate(P), np.concatenate(TVA), np.concatenate(TM)), indent=2))


## 9. Langkah berikutnya
- Ganti dataset ke **Aff-Wild2** (`--layout affwild2`, klip video + anotasi VA `.txt`).
- Serahkan checkpoint `checkpoints/stgcn_va_best.pth` ke modul fusion (kontrak VA).
- Kalibrasi bobot fusi w1,w2 di MAHNOB-HCI / UBFC-Phys (opsional).
